In [3]:
import pathlib
import os
from typing import List

from datasets import load_dataset, load_from_disk
import evaluate
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    print(f"Loading pretrained model from {path}")
    checkpoint = torch.load(path, map_location=device)

    gptconf = GPTConfig(**checkpoint['model_args'])
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint['model']

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)
    pretrained_model.to(device)

    return pretrained_model


# Define variables directly here instead of using argparse
args = {
    'vocab': pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json"),  # Example for IPA
    'merges': pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt"),  # Example for IPA
    'model': pathlib.Path("/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_50k/ckpt.pt"),  # IPA Model
    'task': "mrpc",  # Example task
    'epochs': 3,
    'eval_interval': 0.01,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 8,
    'hf_cache_dir': pathlib.Path('cache'),
    'dataset': 'iggy12345/glue-mrpc-ipa',  # Dataset path
    'from_disk': False,
    'no_subset': False,
    'device': 'cuda',
    'no_progress_bar': False
}

# ---- Models and Tokenizers ----
models_and_tokenizers = [
    {"model_type": "ipa", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt")},
    {"model_type": "normal", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-merges.txt")},
    {"model_type": "prebuilt", "model_path": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt__medium_50k/ckpt.pt",
     "tokenizer_paths": None}
]

# Create a main directory for all model outputs
main_output_dir = pathlib.Path("./training_outputs")
if not os.path.exists(main_output_dir):
    os.makedirs(main_output_dir)

# Loop through the models and tokenizers
for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    tokenizer_paths = config["tokenizer_paths"]

    # Dynamically set the output directory based on the model type
    output_dir = main_output_dir / f"output_{model_type}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # ---- Load Tokenizer ----
    if model_type == "ipa" or model_type == "normal":
        vocab_path, merges_path = tokenizer_paths
        tokenizer = load_tokenizer(vocab_path, merges_path)
    elif model_type == "prebuilt":
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token

    # ---- Load model ----
    base_model = load_pretrained_model(model_path, args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model).to(args['device'])

    # ---- Load dataset ----
    dataset = load_dataset(args['dataset'], cache_dir=str(args['hf_cache_dir']))

    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(example) for example in zip(*[examples[f] for f in features])]

    # ---- Preprocessing ----
    def preprocess_function(examples):
        if 'premise' in examples:
            feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        elif 'question' in examples:
            if 'sentence' in examples:
                feature = flatten_multi_features(examples, ['question', 'sentence'])
            else:
                feature = flatten_multi_features(examples, ['question', 'hypothesis'])
        elif 'sentence1' in examples:
            feature = flatten_multi_features(examples, ['sentence1', 'sentence2'])
        elif 'question1' in examples:
            feature = flatten_multi_features(examples, ['question1', 'question2'])
        else:
            feature = examples['sentence']

        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    encoded_dataset = dataset.map(preprocess_function, batched=True)

    # ---- Data collator ----
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # ---- Metrics ----
    metric = evaluate.load("glue", args['task'])

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        # return metric.compute(predictions=predictions, references=labels)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions)
        hf_metrics["recall"] = recall_score(labels, predictions)
        return hf_metrics

    # ---- Training arguments ----
    training_args = TrainingArguments(
        output_dir=str(output_dir),  # Dynamically set output directory
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",  # Save every 'save_steps'
        save_steps=100,  # Save every 100 steps
        save_total_limit=1,  # Keep only 1 checkpoint
        metric_for_best_model="accuracy",
        load_best_model_at_end=True,  # Automatically load the best model after training
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=100,  # Log every 100 steps
        logging_dir='./logs',  # Log directory to store logs
        disable_tqdm=False,  # Show progress bar in notebook
        warmup_ratio=0.3,  # Try a warmup ratio
        save_safetensors=False  # Disable safetensors saving
    )

    # ---- Trainer ----
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ---- Train ----
    trainer.train(resume_from_checkpoint=False)  
    results = trainer.evaluate(encoded_dataset["validation"])
    print(f"Evaluation results for {model_type}: {results}")


Loading pretrained model from /fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt
number of parameters: 353.24M


/tmp/slurmtmp.1386298/ipykernel_1022559/1873009701.py:171: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
100,0.729900,0.717784,0.566176,0.620985,0.771277,0.519713
200,0.654400,0.574142,0.710784,0.811502,0.731988,0.910394
300,0.555500,0.624591,0.764706,0.843137,0.774775,0.924731
400,0.584300,0.616753,0.727941,0.831563,0.721053,0.982079
500,0.523900,0.568671,0.732843,0.786693,0.866379,0.720430
600,0.440100,0.745421,0.737745,0.837633,0.726316,0.989247
700,0.378400,0.645786,0.772059,0.825516,0.866142,0.788530
800,0.450100,0.548201,0.774510,0.825758,0.875502,0.781362
900,0.391700,0.524298,0.789216,0.853741,0.812298,0.899642
1000,0.153500,0.975259,0.801471,0.858639,0.836735,0.881720


Evaluation results for ipa: {'eval_loss': 1.3581160306930542, 'eval_accuracy': 0.8088235294117647, 'eval_f1': 0.8655172413793103, 'eval_precision': 0.8338870431893688, 'eval_recall': 0.899641577060932, 'eval_runtime': 2.5718, 'eval_samples_per_second': 158.642, 'eval_steps_per_second': 19.83, 'epoch': 3.0}
Loading pretrained model from /fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt
number of parameters: 353.24M


/tmp/slurmtmp.1386298/ipykernel_1022559/1873009701.py:171: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
100,0.736800,0.750636,0.529412,0.573333,0.754386,0.462366
200,0.653500,0.593989,0.723039,0.823161,0.730556,0.942652
300,0.575800,0.530564,0.759804,0.839344,0.773414,0.917563
400,0.589700,0.546767,0.725490,0.827692,0.725067,0.964158
500,0.504100,0.511297,0.764706,0.818868,0.864542,0.777778
600,0.407700,0.779673,0.769608,0.850318,0.765043,0.956989
700,0.405200,0.932093,0.693627,0.739040,0.885000,0.634409
800,0.480500,0.479817,0.816176,0.867257,0.856643,0.878136
900,0.404800,0.462046,0.813725,0.874172,0.812308,0.946237
1000,0.138200,0.947795,0.818627,0.877076,0.817337,0.946237


Evaluation results for normal: {'eval_loss': 0.8653320670127869, 'eval_accuracy': 0.8431372549019608, 'eval_f1': 0.8900343642611683, 'eval_precision': 0.8547854785478548, 'eval_recall': 0.9283154121863799, 'eval_runtime': 4.1779, 'eval_samples_per_second': 97.656, 'eval_steps_per_second': 12.207, 'epoch': 3.0}
Loading pretrained model from /fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt__medium_50k/ckpt.pt


FileNotFoundError: [Errno 2] No such file or directory: '/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt__medium_50k/ckpt.pt'